# Higher Education Institutions and Food System Transformation: The Agricultural Education Paradox in Sub-Saharan Africa Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the **FAIR^2** dataset using the `mlcroissant` library. We will walk through metadata inspection, data loading, basic exploration, and visualization.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.gmwx-gww3/fair2.json](https://sen.science/doi/10.71728/senscience.gmwx-gww3/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.gmwx-gww3/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
meta = dataset.metadata

print(f"Dataset: {meta.name}\n\nDescription: {meta.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs as defined in the Croissant schema.

> **Note:** All references to record sets and fields will use their `@id` values.

In [ ]:
# List available record sets and their fields
record_sets = list(dataset.record_sets)

print(f"Found {len(record_sets)} record set(s) in this dataset.\n")
for recset in record_sets:
    print(f'Record set @id: {recset.id}')
    print(f'  Name        : {recset.name}')
    print(f'  Description : {recset.description}')
    print(f'  Fields:')
    for field in recset.fields:
        print(f'    - @id: {field.id} | name: {field.name} | dataType: {field.data_type}')
    print('-' * 50)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All operations will use the appropriate `@id` keys explicitly.


In [ ]:
# Collect all record set @ids
record_set_ids = [recset.id for recset in dataset.record_sets]
dataframes = {}

# Display available record sets
print(f"Record set IDs: {record_set_ids}\n")

# For demonstration, load all record sets into pandas DataFrames
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"Loaded {len(df)} records from record set '{rec_id}' with columns: {df.columns.tolist()}")

# Choose the main table (if more than one, pick the most complete; here we take first)
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None

if main_record_set_id:
    print(f"\nFirst 5 rows of '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())
else:
    print('No record sets found in this dataset.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, e.g.:
- Filtering records (e.g., years, numeric thresholds).
- Normalizing numeric fields.
- Grouping and summarizing data.

All fields referenced below use their `@id`. Adjust filtering/grouping to meaningful variables from the loaded record set.

In [ ]:
# --- Define field IDs for EDA ---
# To reference fields by @id, examine output from cell 6. Use those values here.

# Example field @ids (replace with actual @id from schema for your data):
# e.g. 'cr:EnrollmentValue', 'cr:CountryCode', etc.

if main_record_set_id:
    main_df = dataframes[main_record_set_id]
    print(f"Columns available for EDA: {main_df.columns.tolist()}")
    # Try to automatically pick numeric field and a group field
    numeric_field_id = None
    group_field_id = None
    # Guess which columns are numeric (float or int)
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break
    # Guess grouping field: something like 'country' or 'indicator'
    for col in main_df.columns:
        if ('coun' in col.lower() or 'indicat' in col.lower() or 'region' in col.lower()) and col != numeric_field_id:
            group_field_id = col
            break
    if numeric_field_id is None:
        import warnings
        warnings.warn('No numeric field detected, EDA steps will be skipped.')
    else:
        # Drop NA for analysis
        df_clean = main_df.dropna(subset=[numeric_field_id])
        threshold = df_clean[numeric_field_id].quantile(0.25)  # Use lower quartile as threshold example
        filtered_df = df_clean[df_clean[numeric_field_id] > threshold]
        print(f"Filtered records in '{main_record_set_id}' where [{numeric_field_id}] > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' (z-score):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group field if possible
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            grouped_df = grouped_df.rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
            print(f"Mean of {numeric_field_id} grouped by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print('No valid group field found.')
else:
    print('Main record set not found; skipping EDA.')

## 5. Visualization
Visualize value distributions and trends. All x/y axes reference field `@id`s, not logical names.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If groupable (e.g. by country indicator/year)
    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(12,6))
        # Show mean per group for top 10 groups with most data
        group_counts = main_df[group_field_id].value_counts().head(10).index
        plot_df = main_df[main_df[group_field_id].isin(group_counts)]
        sns.boxplot(data=plot_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} distribution by '{group_field_id}' (top 10)")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Not enough data for visualization. Make sure columns are detected.')

## 6. Conclusion

- We successfully loaded Croissant metadata and records for the FAIR^2 dataset.
- The schema makes reproducible, FAIR-aligned data access and referencing of fields via `@id` straightforward.
- Basic EDA and field-based visualization reveal the structure and distributions in the data.
- For custom analyses, look up the specific field `@id`s in Section 2 and modify EDA/visuals accordingly.

*For more advanced processing or to cite this dataset, please use the DOI: [10.71728/senscience.gmwx-gww3](https://sen.science/doi/10.71728/senscience.gmwx-gww3).*